In [8]:
import os
import requests
import json
from typing import List, Dict
import time
import numpy as np

import duckdb

MISTRAL = os.getenv("MISTRAL_AI")

In [9]:
def carregar_comentarios_de_txt(filename: str = "comentarios.txt") -> List[str]:
    """Carrega os comentários de um arquivo de texto e os retorna em uma lista."""
    comentarios_carregados = []
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for linha in f:
                comentarios_carregados.append(linha.strip())
        print(f"Comentários carregados com sucesso do arquivo '{filename}'")
    except FileNotFoundError:
        print(f"Arquivo '{filename}' não encontrado.")
    except IOError as e:
        print(f"Erro ao ler o arquivo '{filename}': {e}")
    return comentarios_carregados

In [10]:
comentarios = carregar_comentarios_de_txt()

Comentários carregados com sucesso do arquivo 'comentarios.txt'


In [11]:
import requests
import json
import time
from typing import List, Optional
from datetime import datetime, timedelta

class MistralEmbedder:
    def __init__(self, api_key: str, max_retries: int = 5, initial_delay: float = 3.0):
        self.api_key = api_key
        self.max_retries = max_retries
        self.initial_delay = initial_delay
        self.last_request_time = None
        self.min_request_interval = timedelta(seconds=1)  # Intervalo mínimo entre requisições

    def _make_request(self, comment: str, current_retry: int = 0) -> Optional[List[float]]:
        """Faz uma requisição individual com tratamento de erros e retentativas"""
        url = "https://api.mistral.ai/v1/embeddings"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}"
        }
        payload = {
            "model": "mistral-embed",
            "input": comment
        }

        # Respeita o intervalo mínimo entre requisições
        if self.last_request_time is not None:
            elapsed = datetime.now() - self.last_request_time
            if elapsed < self.min_request_interval:
                wait_time = (self.min_request_interval - elapsed).total_seconds()
                time.sleep(wait_time)

        try:
            response = requests.post(url, headers=headers, json=payload)
            self.last_request_time = datetime.now()

            # Verifica rate limiting (429 - Too Many Requests)
            if response.status_code == 429:
                retry_after = float(response.headers.get('Retry-After', self.initial_delay * (2 ** current_retry)))
                print(f"Rate limit atingido. Tentando novamente em {retry_after} segundos...")
                time.sleep(retry_after)
                return self._make_request(comment, current_retry + 1)

            response.raise_for_status()

            embeddings_data = response.json()
            if (embeddings_data and "data" in embeddings_data and 
                len(embeddings_data["data"]) > 0 and 
                "embedding" in embeddings_data["data"][0]):
                return embeddings_data["data"][0]["embedding"]

            print(f"Resposta inesperada para: '{comment[:50]}...'")
            return None

        except requests.exceptions.RequestException as e:
            print(f"Erro na requisição (tentativa {current_retry + 1}/{self.max_retries}): {e}")
            if current_retry < self.max_retries - 1:
                delay = self.initial_delay * (2 ** current_retry)  # Exponential backoff
                print(f"Esperando {delay} segundos antes de tentar novamente...")
                time.sleep(delay)
                return self._make_request(comment, current_retry + 1)
            print(f"Falha após {self.max_retries} tentativas para: '{comment[:50]}...'")
            return None

    def create_embeddings(self, comments: List[str]) -> List[List[float]]:
        """Cria embeddings para uma lista de comentários com tratamento robusto"""
        embeddings = []
        total = len(comments)
        
        for i, comment in enumerate(comments, 1):
            print(f"Processando comentário {i}/{total}...")
            embedding = self._make_request(comment)
            if embedding is not None:
                embeddings.append(embedding)
            else:
                embeddings.append([])  # Mantém o alinhamento com a lista original

        return embeddings

In [12]:
# Inicializa o embedder
embedder = MistralEmbedder(api_key=MISTRAL)

# Cria os embeddings
embeddings_list = embedder.create_embeddings(comentarios)

Processando comentário 1/1221...
Processando comentário 2/1221...
Processando comentário 3/1221...
Processando comentário 4/1221...
Processando comentário 5/1221...
Processando comentário 6/1221...
Processando comentário 7/1221...
Processando comentário 8/1221...
Processando comentário 9/1221...
Processando comentário 10/1221...
Rate limit atingido. Tentando novamente em 3.0 segundos...
Processando comentário 11/1221...
Processando comentário 12/1221...
Processando comentário 13/1221...
Processando comentário 14/1221...
Processando comentário 15/1221...
Processando comentário 16/1221...
Processando comentário 17/1221...
Processando comentário 18/1221...
Processando comentário 19/1221...
Processando comentário 20/1221...
Processando comentário 21/1221...
Rate limit atingido. Tentando novamente em 3.0 segundos...
Processando comentário 22/1221...
Processando comentário 23/1221...
Processando comentário 24/1221...
Processando comentário 25/1221...
Processando comentário 26/1221...
Process

In [13]:
len(embeddings_list)

1221

In [15]:
with open("comments_1221_mistral.json", 'w', encoding='utf-8') as f:
    json.dump(embeddings_list, f, indent=4)
    
print(f"Dados exportados com sucesso para JSON")

Dados exportados com sucesso para JSON


In [26]:
con = duckdb.connect("comments.duckdb")
con.execute("""
    CREATE TABLE IF NOT EXISTS comments (
        id INTEGER,
        text STRING,
        vec FLOAT[]
    )
""")

# Inserindo os comentários
for id, (comment, embedding) in enumerate(zip(comentarios, embeddings_list)):
    print(f"id: {id}, comment {comment[:50]}, emb {embedding[:3]}")
    
    con.execute("""
    INSERT OR IGNORE INTO comments
    VALUES (?, ?, ?)
    """, [id, comment, embedding])

con.close()

id: 0, comment 3:03 "You can clap about that all you want. Enjoy", emb [-0.0268707275390625, 0.0340576171875, 0.017730712890625]


BinderException: Binder Error: There are no UNIQUE/PRIMARY KEY Indexes that refer to this table, ON CONFLICT is a no-op

In [ ]:
DATABASE_FILE = "comments.duckdb"
JSON_OUTPUT_FILE = "comments.json"

def exportar_para_json(database_file: str = DATABASE_FILE, json_file: str = JSON_OUTPUT_FILE):
    con = duckdb.connect(database_file)
    try:
        # Consultar todos os dados da tabela
        result = con.execute("SELECT text, vec FROM comments").fetchall()
        data_para_json = []
        for row in result:
            data_para_json.append({"text": row[0], "embedding": row[1]})

        # Salvar os dados em um arquivo JSON
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(data_para_json, f, indent=4)
        print(f"Dados exportados com sucesso para JSON em '{json_file}'")
    except duckdb.CatalogException as e:
        print(f"Erro ao exportar para JSON: {e}")
    finally:
        con.close()
        
exportar_para_json()

Erro ao exportar para JSON: Catalog Error: Table with name comments does not exist!
Did you mean "information_schema.columns"?

LINE 1: SELECT text, vec FROM comments
                              ^


In [ ]:
# Buscar comentários que contenham determinada palavra
with duckdb.connect("comments.duckdb") as con:
    search_term = "hurry"
    result = con.execute("""
        SELECT text 
        FROM comments 
        WHERE text LIKE '%' || ? || '%'
    """, [search_term]).fetchdf()  # Retorna como DataFrame

In [ ]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = sum(x * x for x in a) ** 0.5
    magnitude_b = sum(x * x for x in b) ** 0.5
    return dot_product / (magnitude_a * magnitude_b)

def find_similar_to_reference(reference_emb, all_comments, comment_embeddings, top_n=10):
    """
    Encontra comentários mais similares ao embedding de referência
    
    Args:
        reference_emb: List/Array - Embedding de referência
        all_comments: List[str] - Lista de textos de comentários
        comment_embeddings: List[List[float]] - Embeddings correspondentes
        top_n: int - Número de resultados a retornar
        
    Returns:
        List[Tuple[str, float]] - (comentário, similaridade)
    """
    similarities = [
        (comment, cosine_similarity(reference_emb, emb))
        for comment, emb in zip(all_comments, comment_embeddings)
    ]
    
    # Ordena por similaridade (maior primeiro) e pega os top_n
    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_n]

In [ ]:
def carregar_embedding(filename: str = "embedding.json") -> list:
    """Carrega um embedding de um arquivo JSON.

    Args:
        filename: O nome do arquivo JSON do qual carregar o embedding.

    Returns:
        Uma lista de floats representando o embedding, ou None se ocorrer um erro.
    """
    try:
        with open(filename, 'r') as f:
            data = json.load(f)
            if "embedding" in data and isinstance(data["embedding"], list) and all(isinstance(item, float) for item in data["embedding"]):
                return data["embedding"]
            else:
                print(f"Formato inválido no arquivo '{filename}'. Esperava uma lista de floats na chave 'embedding'.")
                return None
    except FileNotFoundError:
        print(f"Arquivo '{filename}' não encontrado.")
        return None
    except json.JSONDecodeError:
        print(f"Erro ao decodificar JSON do arquivo '{filename}'.")
        return None
    except IOError as e:
        print(f"Erro ao ler o arquivo '{filename}': {e}")
        return None

In [ ]:
def get_embeddings_from_db():
    """Busca todos os comentários e embeddings do banco de dados"""
    with duckdb.connect("comments.duckdb") as con:
    
        # Busca todos os comentários
        query = "SELECT comment_id, text, vec FROM comments"
        data = con.execute(query).fetchall()
    
        if not data:
            return [], [], []
            
        # Desempacota os resultados
        ids, texts, embeddings = zip(*data)
        return list(ids), list(texts), list(embeddings)

In [ ]:
def find_similar_comments_db(reference_embedding, top_n=5):
    """
    Busca comentários similares diretamente do banco de dados
    
    Args:
        reference_embedding: Embedding de referência (lista/array)
        video_id: Filtrar por vídeo específico (opcional)
        top_n: Quantos resultados retornar
    """
    # Limpa o embedding de referência
    clean_ref = [x for x in reference_embedding if isinstance(x, (int, float))]
    
    # Busca dados do banco
    ids, texts, embeddings = get_embeddings_from_db()
    
    # Calcula similaridades
    results = []
    for comment_id, text, emb in zip(ids, texts, embeddings):
        try:
            # Limpa o embedding do banco
            clean_emb = [x for x in emb if isinstance(x, (int, float))]
            
            # Calcula similaridade
            similarity = cosine_similarity(clean_ref, clean_emb)
            results.append((comment_id, text, similarity))
        except Exception as e:
            print(f"Erro ao processar comentário {comment_id}: {str(e)}")
            continue
    
    # Ordena e retorna os top_n
    return sorted(results, key=lambda x: x[2], reverse=True)[:top_n]

In [ ]:
emb = carregar_embedding() # embedding criado externamente - short_emb.ipynb

reference_emb = emb  # Seu embedding aqui
similar_comments = find_similar_comments_db(reference_emb, top_n=3)

In [ ]:
similar_comments

[('UgyxqbJblUyi7_1p2ad4AaABAg',
  'Very, very hard to shake of the feeling how manipulative and insecure Altman is. I really picked it up from the first conversation I saw him in. He is in this for himself only and thus finds it hard to constantly give the opposing impression in interviews. Concerning times… The disruption is imminent and the CEO of the leading AI company is too insecure to answer tough, honest questions… I must count on people organising against this ”AI revolution”. Else this will touch millions of careers in the matter of couple years.',
  0.8336400418180048),
 ('Ugx-gHu0U9SoAQwCKqJ4AaABAg',
  'Just watching  the TED2025 interview with Sam Altman. A masterclass in missing the point. Fear-mongering about AI apocalypse, token gestures toward artist rights, and absolutely no engagement with the real power dynamics at play. When critical questions are needed most, we get soundbites. Disappointing.',
  0.8011777949717447),
 ('UgxTd8tNrSX3_BAMknR4AaABAg',
  'Claude\'s Cri

In [ ]:
# # Delete Table


# import duckdb

# Conectar ao banco de dados DuckDB (pode ser um arquivo ou in-memory)
# con = duckdb.connect('comments.duckdb')  # Para um arquivo
# con = duckdb.connect(':memory:')     # Para um banco de dados in-memory

# try:
#     Executar o comando DROP TABLE
#     con.execute("DROP TABLE comments;")
#     print("Tabela 'minha_tabela' deletada com sucesso.")
# except duckdb.CatalogException as e:
#     print(f"Erro ao deletar a tabela: {e}")
# finally:
#     Fechar a conexão
#     con.close()

Tabela 'minha_tabela' deletada com sucesso.
